# 🍜 EDA · Mercado publicitario en el sector gastronómico

> **Dataset primario de auditoría** para análisis de mercado publicitario en el sector gastronómico. Contiene registros crudos (*raw data*) con anomalías simuladas del sector (duplicados, valores nulos y registros vacíos) para evaluar la resiliencia de los flujos de limpieza (ETL) en Python y su posterior ingesta para optimización de presupuestos de pauta y segmentación geoespacial.

| Sesión | Alcance | Estado |
|---|---|---|
| **1** | Estructura del notebook, datasets crudos (CSV + MD), selector de fuente, diagnóstico y limpieza base | ✅ Hecho |
| **2** | Modelo relacional, carga a SQLite (`restaurantes.db`), consultas de negocio una por una | 🔄 En curso |
| **3** | Iteraciones de imputación, empalme con datos reales, export para Tableau | ⏳ Pendiente |

📒 Bitácora de negocio y gobernanza: `E:\saoto\Documents\Portafolio\bitacora_restaurantes_publicidad.md`

## 0 · Arquitectura del proyecto (brújula)

```
Portafolio/
├── bitacora_restaurantes_publicidad.md   ← hoja de respuestas externa (negocio + gobernanza)
└── contexto-analisis/
    ├── EDA_Restaurantes_Publicidad.ipynb ← este notebook (guía maestra + ETL)
    ├── restaurantes_raw.csv / .md        ← Hoja 1 · catálogo (con anomalías)
    ├── campana_sprints_raw.csv / .md     ← Hoja 2 · campaña 4 sprints
    ├── sql/
    │   ├── 01_schema.sql                 ← modelo relacional (CREATE TABLE)
    │   ├── 02_vistas.sql                 ← vistas: v_restaurante, v_kpis_sprint
    │   └── consultas/QNN_*.sql           ← una consulta de negocio por archivo
    ├── restaurantes.db                   ← base SQLite generada por el notebook (no se versiona)
    └── salidas/                          ← (Sesión 3) datos limpios para Tableau
```

### Hoja 1 · Catálogo (`restaurantes_raw`)

| Columna | Tipo | Descripción |
|---|---|---|
| `id_restaurante` | str | Llave primaria (`R001`…). Une Hoja 1 ↔ Hoja 2 |
| `nombre` | str | Nombre comercial |
| `zona` | str | Zona de influencia (Centro, Norte, Sur, Oriente, Poniente) — *simulada, por validar* |
| `categoria` | str | Tipo de cocina / formato |
| `seguidores_instagram` | int | Seguidores de la cuenta (rango simulado 10k–50k) |
| `engagement_rate` | float | Engagement en **%** (4.1 = 4.1 %) |
| `url_instagram` | str | Llave de empalme con los datos reales de IG |

**Anomalías sembradas a propósito:** espacios al borde (`nombre`, `zona`, `categoria`), 1 duplicado exacto (R002), 1 duplicado que solo aparece tras limpiar espacios (R005), 1 registro totalmente vacío, nulos en `engagement_rate` (R006, R009) y `seguidores_instagram` (R007).

### Hoja 2 · Seguimiento de campaña (`campana_sprints_raw`)
1 mes de pauta (oct-2026) = 4 sprints semanales × 10 restaurantes = 40 filas.

| Grupo | Columnas |
|---|---|
| Identificación | `id_restaurante`, `sprint` (S1–S4), `fecha_inicio`, `fecha_fin` |
| Ubicación | `ciudad`, `pais` (atributos del restaurante → se mueven a su tabla en la sección 6) |
| Entradas (se capturan) | `inversion_pauta`, `impresiones`, `alcance`, `clics`, `conversiones`, `ticket_promedio`, `costo_variable_pct` |
| KPIs (se derivan) | `ctr`, `tasa_conversion`, `cpa`, `ingresos_atribuidos`, `margen_contribucion`, `roas`, `roi_publicitario` |

**Fórmulas de negocio**

- `ctr = clics / impresiones`
- `tasa_conversion = conversiones / clics`
- `cpa = inversion_pauta / conversiones`
- `ingresos_atribuidos = conversiones × ticket_promedio`
- `margen_contribucion = ingresos_atribuidos × (1 − costo_variable_pct) − inversion_pauta`
- `roas = ingresos_atribuidos / inversion_pauta`
- `roi_publicitario = margen_contribucion / inversion_pauta`

> Al reemplazar datos simulados por reales basta con capturar las **entradas**; la sección 5 recalcula y audita los KPIs.

## 1 · Configuración y selector de fuente

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

BASE = Path.cwd()

# 🔀 Selector de fuente: "csv" o "md"
FUENTE = "csv"

COLS_NUM_H1 = ["seguidores_instagram", "engagement_rate"]
COLS_TXT_H1 = ["id_restaurante", "nombre", "zona", "categoria", "url_instagram"]

COLS_NUM_H2 = ["inversion_pauta", "impresiones", "alcance", "clics", "conversiones",
               "ticket_promedio", "costo_variable_pct", "ctr", "tasa_conversion", "cpa",
               "ingresos_atribuidos", "margen_contribucion", "roas", "roi_publicitario"]
COLS_TXT_H2 = ["id_restaurante", "sprint", "ciudad", "pais"]

## 2 · Carga de datos (CSV ↔ Markdown)

Ambos lectores leen todo como texto **sin limpiar** (para que las anomalías sobrevivan a la carga) y luego convierten las columnas numéricas con la misma regla.

In [ ]:
def leer_csv(ruta: Path) -> pd.DataFrame:
    # dtype=str conserva los espacios; las celdas vacías llegan como NaN
    return pd.read_csv(ruta, dtype=str, skip_blank_lines=True)


def leer_md(ruta: Path) -> pd.DataFrame:
    """Lee la primera tabla Markdown del archivo.
    Cada celda trae exactamente 1 espacio de relleno a cada lado; lo demás es dato."""
    filas = [l.rstrip("\n") for l in ruta.read_text(encoding="utf-8").splitlines()
             if l.startswith("|")]
    def celdas(linea):
        return [c[1:-1] if len(c) >= 2 else "" for c in linea[1:-1].split("|")]
    encabezado = [c.strip() for c in celdas(filas[0])]
    datos = [celdas(l) for l in filas[2:]]  # filas[1] es el separador |---|
    return pd.DataFrame(datos, columns=encabezado).replace("", np.nan)


def tipar(df: pd.DataFrame, cols_num) -> pd.DataFrame:
    df = df.copy()
    for c in cols_num:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def cargar(nombre_base: str, cols_num, fuente: str = FUENTE) -> pd.DataFrame:
    lectores = {"csv": leer_csv, "md": leer_md}
    if fuente not in lectores:
        raise ValueError(f"FUENTE debe ser 'csv' o 'md', no {fuente!r}")
    return tipar(lectores[fuente](BASE / f"{nombre_base}.{fuente}"), cols_num)


df_raw = cargar("restaurantes_raw", COLS_NUM_H1)
print(f"Fuente: {FUENTE} · {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
df_raw

### 2.1 · Prueba de paridad: ¿Python procesa ambas fuentes de forma indistinta?

In [ ]:
df_csv = cargar("restaurantes_raw", COLS_NUM_H1, "csv")
df_md = cargar("restaurantes_raw", COLS_NUM_H1, "md")
pd.testing.assert_frame_equal(df_csv, df_md)
pd.testing.assert_frame_equal(cargar("campana_sprints_raw", COLS_NUM_H2, "csv"),
                              cargar("campana_sprints_raw", COLS_NUM_H2, "md"))
print("✅ CSV y Markdown producen DataFrames idénticos en ambas hojas (valores, tipos y espacios).")

## 3 · Diagnóstico de calidad (antes de limpiar)

In [ ]:
def diagnostico(df: pd.DataFrame) -> pd.Series:
    txt = df.select_dtypes(include="object")
    con_espacios = txt.apply(lambda s: s.str.len().ne(s.str.strip().str.len())).sum().sum()
    return pd.Series({
        "filas": len(df),
        "filas_totalmente_vacias": int(df.isna().all(axis=1).sum()),
        "duplicados_exactos": int(df.duplicated().sum()),
        "celdas_con_espacios_al_borde": int(con_espacios),
        "nulos_seguidores_instagram": int(df["seguidores_instagram"].isna().sum()),
        "nulos_engagement_rate": int(df["engagement_rate"].isna().sum()),
    })

diag_antes = diagnostico(df_raw)
diag_antes

In [ ]:
# Detalle de nulos por columna
df_raw.isna().sum().to_frame("nulos")

## 4 · Limpieza ETL (Hoja 1)

Orden deliberado: **espacios → vacíos → duplicados → nulos**. Si se deduplica antes de recortar espacios, el duplicado de R005 (`"Helados Clemente  "`) pasa desapercibido.

In [ ]:
df = df_raw.copy()

# 4.1 Recortar espacios al borde en columnas de texto
for c in COLS_TXT_H1:
    df[c] = df[c].str.strip()

# 4.2 Eliminar registros totalmente vacíos
df = df.dropna(how="all")

# 4.3 Eliminar duplicados (tras normalizar texto)
df = df.drop_duplicates()
assert df["id_restaurante"].is_unique, "Hay id_restaurante repetidos con datos distintos: revisar a mano"

df.shape

### 4.4 · Gestión de nulos

Decisión de Sesión 1 (provisional, documentar en bitácora): **no perder restaurantes** del catálogo. Se imputa con la mediana de la categoría y, si no existe, la mediana global; una columna bandera deja rastro de qué valor es imputado.

> 🔁 Sesión 2: comparar contra `dropna()` y contra empalme con datos reales antes de fijar la regla.

In [ ]:
for c in COLS_NUM_H1:
    df[f"{c}_imputado"] = df[c].isna()
    mediana_cat = df.groupby("categoria")[c].transform("median")
    df[c] = df[c].fillna(mediana_cat).fillna(df[c].median())

df["seguidores_instagram"] = df["seguidores_instagram"].round().astype("int64")
df = df.sort_values("id_restaurante").reset_index(drop=True)
df_limpio = df
df_limpio

In [ ]:
pd.DataFrame({"antes": diag_antes, "después": diagnostico(df_limpio.drop(columns=[c for c in df_limpio if c.endswith("_imputado")]))})

## 5 · Hoja 2 · Campaña a 4 sprints (carga + auditoría de KPIs)

In [ ]:
camp = cargar("campana_sprints_raw", COLS_NUM_H2)
for c in COLS_TXT_H2:
    camp[c] = camp[c].str.strip()
camp[["fecha_inicio", "fecha_fin"]] = camp[["fecha_inicio", "fecha_fin"]].apply(pd.to_datetime)
print(camp.shape)
camp.head(8)

In [ ]:
def calcular_kpis(d: pd.DataFrame) -> pd.DataFrame:
    k = pd.DataFrame(index=d.index)
    k["ctr"] = d["clics"] / d["impresiones"]
    k["tasa_conversion"] = d["conversiones"] / d["clics"]
    k["cpa"] = d["inversion_pauta"] / d["conversiones"]
    k["ingresos_atribuidos"] = d["conversiones"] * d["ticket_promedio"]
    k["margen_contribucion"] = k["ingresos_atribuidos"] * (1 - d["costo_variable_pct"]) - d["inversion_pauta"]
    k["roas"] = k["ingresos_atribuidos"] / d["inversion_pauta"]
    k["roi_publicitario"] = k["margen_contribucion"] / d["inversion_pauta"]
    return k

kpis = calcular_kpis(camp)
# Tolerancia relativa del 1 % por redondeos en la captura
desvios = ~np.isclose(camp[kpis.columns], kpis, rtol=0.01)
print("Filas con KPI inconsistente:", int(desvios.any(axis=1).sum()))

In [ ]:
# Integridad referencial Hoja 2 → Hoja 1
huerfanos = set(camp["id_restaurante"]) - set(df_limpio["id_restaurante"])
print("IDs de campaña sin catálogo:", huerfanos or "ninguno ✅")

## 6 · Modelo relacional

Las dos hojas mezclan varias entidades. Se separan en un **esquema estrella** (DDL completo en `sql/01_schema.sql`):

```
 pais (1) ──< ciudad (1) ──< restaurante >── (1) categoria
                                  │
                                 (1)
                                  ^
                          metricas_sprint >── (1) sprint
```

| Tabla | Llave | Viene de | Decisión de normalización |
|---|---|---|---|
| `pais` | `id_pais` | Hoja 2 | Se repite en 40 filas → catálogo propio |
| `ciudad` | `id_ciudad` (FK `id_pais`) | Hoja 2 | **3FN**: `pais` depende de `ciudad`, no del restaurante |
| `categoria` | `id_categoria` | Hoja 1 | Catálogo para agrupar |
| `restaurante` | `id_restaurante` | Hoja 1 + Hoja 2 | Recibe `ciudad` y `costo_variable_pct` (**2FN**: dependen solo del restaurante, no del sprint) |
| `sprint` | `id_sprint` | Hoja 2 | **2FN**: las fechas dependen solo del sprint |
| `metricas_sprint` | `id_restaurante + id_sprint` | Hoja 2 | Solo **entradas**; los KPIs se calculan en la vista `v_kpis_sprint` |

`zona` queda como columna de `restaurante`: "Centro" de Miami y "Centro" de Bogotá no son la misma zona.

### 6.1 · Verificar las dependencias antes de separar
Si alguna falla, el dato no puede moverse a esa tabla. Por ejemplo: si `costo_variable_pct` cambiara entre sprints, tendría que quedarse en `metricas_sprint`.

In [ ]:
def depende_de(d: pd.DataFrame, llave, cols) -> bool:
    """True si cada valor de `llave` tiene un único valor en `cols` (dependencia funcional)."""
    return bool((d.groupby(llave)[cols].nunique(dropna=False) <= 1).all().all())

dependencias = pd.Series({
    "id_restaurante → ciudad, pais, costo_variable_pct": depende_de(camp, "id_restaurante", ["ciudad", "pais", "costo_variable_pct"]),
    "sprint → fecha_inicio, fecha_fin": depende_de(camp, "sprint", ["fecha_inicio", "fecha_fin"]),
    "ciudad → pais": depende_de(camp, "ciudad", ["pais"]),
})
assert dependencias.all(), "Una dependencia no se cumple: revisar el modelo antes de cargar"
dependencias

### 6.2 · Construir las tablas en pandas

In [ ]:
def dimension(valores: pd.Series, col_id: str) -> pd.DataFrame:
    """Catálogo con ID entero 1..n a partir de valores únicos (orden alfabético)."""
    unicos = sorted(valores.dropna().unique())
    return pd.DataFrame({col_id: range(1, len(unicos) + 1), "nombre": unicos})

# Atributos del restaurante que llegaron en la Hoja 2 (1 fila por restaurante)
ubicacion = camp[["id_restaurante", "ciudad", "pais", "costo_variable_pct"]].drop_duplicates()

t_pais = dimension(ubicacion["pais"], "id_pais")

ciudades = (ubicacion[["ciudad", "pais"]].drop_duplicates()
            .merge(t_pais.rename(columns={"nombre": "pais"}), on="pais")
            .sort_values(["pais", "ciudad"]).reset_index(drop=True))
ciudades.insert(0, "id_ciudad", ciudades.index + 1)
t_ciudad = ciudades.rename(columns={"ciudad": "nombre"})[["id_ciudad", "nombre", "id_pais"]]

t_categoria = dimension(df_limpio["categoria"], "id_categoria")

t_restaurante = (df_limpio
    .merge(ubicacion, on="id_restaurante", how="left", validate="one_to_one")
    .merge(ciudades[["id_ciudad", "ciudad", "pais"]], on=["ciudad", "pais"], how="left")
    .merge(t_categoria.rename(columns={"nombre": "categoria"}), on="categoria", how="left")
    [["id_restaurante", "nombre", "zona", "id_ciudad", "id_categoria",
      "seguidores_instagram", "engagement_rate", "costo_variable_pct", "url_instagram",
      "seguidores_instagram_imputado", "engagement_rate_imputado"]])
assert t_restaurante[["id_ciudad", "id_categoria", "costo_variable_pct"]].notna().all().all(), \
    "Hay restaurantes del catálogo sin campaña (sin ciudad / costo variable)"

t_sprint = (camp[["sprint", "fecha_inicio", "fecha_fin"]].drop_duplicates()
            .rename(columns={"sprint": "id_sprint"})
            .sort_values("fecha_inicio").reset_index(drop=True))
t_sprint.insert(1, "numero", t_sprint.index + 1)
for c in ["fecha_inicio", "fecha_fin"]:
    t_sprint[c] = t_sprint[c].dt.strftime("%Y-%m-%d")  # SQLite guarda fechas como texto ISO

ENTRADAS = ["inversion_pauta", "impresiones", "alcance", "clics", "conversiones", "ticket_promedio"]
t_metricas = camp.rename(columns={"sprint": "id_sprint"})[["id_restaurante", "id_sprint", *ENTRADAS]]

# Orden de carga: dimensiones (padres) → hechos (hijos)
tablas = {
    "pais": t_pais,
    "ciudad": t_ciudad,
    "categoria": t_categoria,
    "restaurante": t_restaurante,
    "sprint": t_sprint,
    "metricas_sprint": t_metricas,
}
pd.Series({k: v.shape for k, v in tablas.items()}, name="filas × columnas")

In [ ]:
t_restaurante

## 7 · Carga a SQLite (`restaurantes.db`)

1. `01_schema.sql` borra y recrea las tablas, así que se puede volver a ejecutar sin problema.
2. `to_sql(..., if_exists="append")` inserta los datos en las tablas ya creadas, así se respetan sus PK, FK y CHECK.
3. `02_vistas.sql` crea `v_restaurante` y `v_kpis_sprint`.

In [ ]:
import sqlite3

SQL = BASE / "sql"
DB = BASE / "restaurantes.db"

con = sqlite3.connect(DB)
con.execute("PRAGMA foreign_keys = ON")
con.executescript((SQL / "01_schema.sql").read_text(encoding="utf-8"))

for nombre, t in tablas.items():
    t.to_sql(nombre, con, if_exists="append", index=False)

con.executescript((SQL / "02_vistas.sql").read_text(encoding="utf-8"))
con.commit()
print(f"✅ Base cargada en {DB.name}")

### 7.1 · Validación de la carga

In [ ]:
conteos = pd.DataFrame({
    "pandas": {k: len(v) for k, v in tablas.items()},
    "sqlite": {k: con.execute(f"SELECT COUNT(*) FROM {k}").fetchone()[0] for k in tablas},
})
huerfanos_fk = pd.read_sql("PRAGMA foreign_key_check", con)
assert conteos["pandas"].equals(conteos["sqlite"]), "Los conteos no coinciden"
assert huerfanos_fk.empty, "Hay filas que apuntan a una llave foránea inexistente"

# Los KPIs de la vista SQL deben coincidir con los que traía el CSV
v = pd.read_sql("SELECT * FROM v_kpis_sprint ORDER BY id_restaurante, sprint_num", con)
ref = camp.sort_values(["id_restaurante", "sprint"]).reset_index(drop=True)
kpis_ok = np.isclose(v[kpis.columns], ref[kpis.columns], rtol=0.01).all()
assert kpis_ok, "La vista v_kpis_sprint no reproduce los KPIs del archivo"

print("✅ Conteos iguales · sin huérfanos de FK · KPIs de la vista = KPIs del archivo")
conteos

## 8 · Consultas de negocio (una por una)

Cada pregunta se guarda en `sql/consultas/QNN_nombre.sql` con su ficha (❓ pregunta, 🔍 detalle, 📊 métricas, 🏷️ agrupación) y se ejecuta aquí. El resultado y la lectura de negocio se anotan en la bitácora.

In [ ]:
def consulta(archivo: str, **params) -> pd.DataFrame:
    """Ejecuta sql/consultas/<archivo> contra restaurantes.db."""
    texto = (SQL / "consultas" / archivo).read_text(encoding="utf-8")
    return pd.read_sql(texto, con, params=params or None)

consulta("Q00_plantilla.sql")

## 9 · Export para Tableau — ⏳ Sesión 3

- Exportar `v_restaurante` y `v_kpis_sprint` a `salidas/` (o conectar Tableau directo a los CSV).
- Pendiente para el mapa: `latitud` / `longitud` por restaurante.

In [ ]:
# ⏳ Sesión 3
# SALIDAS = BASE / "salidas"; SALIDAS.mkdir(exist_ok=True)
# pd.read_sql("SELECT * FROM v_restaurante", con).to_csv(SALIDAS / "restaurantes.csv", index=False)
# pd.read_sql("SELECT * FROM v_kpis_sprint", con).to_csv(SALIDAS / "kpis_sprint.csv", index=False)

## ✅ Checklist

- [x] Dataset crudo Hoja 1 (CSV + MD) con anomalías
- [x] Dataset Hoja 2 (4 sprints) con KPIs de negocio, ciudad y país
- [x] Selector de fuente y prueba de paridad CSV ↔ MD (ambas hojas)
- [x] Diagnóstico y limpieza base (espacios, vacíos, duplicados, nulos)
- [x] Modelo relacional normalizado + carga a SQLite validada
- [ ] Consultas de negocio Q01, Q02… (una por una)
- [ ] Empalmar datos reales de Instagram (seguidores, engagement) por `url_instagram`
- [ ] Validar zonas reales y agregar coordenadas
- [ ] Sesión 3: iteraciones de imputación, export a Tableau